*0.2 Math / ML basics*

# Matrix multiplication

**The situation.** Search works for one query against 4,000 articles. Now the product team wants "related articles" on every page: 4,000 queries, each against 4,000 articles — 16 million scores, rebuilt nightly. A Python loop over dot products takes 20 minutes. The same job as one matrix multiplication takes well under a second.

**Matrix multiplication.** Stack the query vectors as rows of one matrix and the article vectors as rows of another. `queries @ articles.T` computes every query-article dot product in one call, and NumPy hands the work to optimised, multi-core code. This one operation is also what a neural network is made of — every layer is a matrix multiplication.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Time the two ways.** 2,000 unit-length vectors on each side — random here, so the notebook does not spend on 4,000 embeddings; the timing is identical with real ones.

In [2]:
import time

import numpy as np
from sklearn.preprocessing import normalize

rng = np.random.default_rng(0)
articles = normalize(rng.standard_normal((2_000, 1536)).astype(np.float32))
queries = normalize(rng.standard_normal((2_000, 1536)).astype(np.float32))

started = time.perf_counter()
loop_scores = np.empty(
    (200, 2_000), dtype=np.float32
)  # only 200 queries — the full loop is too slow to wait for
for row, query in enumerate(queries[:200]):
    for column, article in enumerate(articles):
        loop_scores[row, column] = query @ article
loop_seconds = (time.perf_counter() - started) * 10  # scaled up to 2,000 queries

started = time.perf_counter()
matrix_scores = queries @ articles.T  # every query against every article
matmul_seconds = time.perf_counter() - started

print("Python loop (2,000 × 2,000, estimated):", round(loop_seconds, 1), "s")
print("matrix multiplication:                 ", round(matmul_seconds, 3), "s")
print(
    "result shape:",
    matrix_scores.shape,
    "| same numbers:",
    np.allclose(loop_scores, matrix_scores[:200], atol=1e-4),
)
assert matmul_seconds * 20 < loop_seconds

Python loop (2,000 × 2,000, estimated): 2.6 s
matrix multiplication:                  0.012 s
result shape: (2000, 2000) | same numbers: True


**Reading the output.** The same 4 million scores, hundreds of times faster, with identical numbers. The shape `(2000, 2000)` reads as: row = query, column = article, cell = their score.

```
queries  (2000 × 1536)   @   articles.T  (1536 × 2000)   =   scores  (2000 × 2000)
   one row per query          one column per article          scores[i, j] = query i · article j
```

**Top-3 related articles for every query** comes straight from the score matrix.

In [3]:
top3 = np.argsort(-matrix_scores, axis=1)[:, :3]
print(
    "related articles for query 0:", top3[0], "with scores", np.round(matrix_scores[0, top3[0]], 3)
)
assert top3.shape == (2_000, 3)

related articles for query 0: [1660 1153 1502] with scores [0.079 0.078 0.077]


**The rule to remember.** If you are looping over vectors, you are doing a matrix multiplication slowly. Stack them and use `@`.

| Use it when | Don't when | Instead use |
|---|---|---|
| many-against-many scoring, batch inference, anything with vectors in a loop | the matrix does not fit in memory (1M × 1M scores = 4 TB) | an approximate index (HNSW) that avoids computing every pair |

**Watch out**
- The inner sizes must match: `(2000 × 1536) @ (1536 × 2000)` works; forgetting `.T` raises a shape error — the friendliest bug in this repo.
- On a GPU the same line is another 50× faster; that is why model serving runs on GPUs.
- A score matrix grows with the square of the number of items; check the memory before building it.